# ECE 3715 Mini-Project 2

## Continuous Random Variables
Professor: Dr. Nir Regev

Team Members: Brandon Chie, Octavio Reyna, William Hsu

Start Date: October 28, 2026

Due Date: November 6, 2026

## Objective
Use continuous probability models and simulations to study random-variable transformations, moments, the central limit theorem, and signal-detection thresholds.

In [1]:
import time #Used for the timer function
import numpy #Used for easy array manipulation functions & RNG
import scipy #Used for theoretical probabilities of binomial, poisson and geometric distribution
import matplotlib.pyplot as plt #Used for plotting graphs
import mp2lib as func #Functions from mp2lib.py file

T_START = time.time()     #Timer begins for total notebook time

SEED = 993409308          #Drawn 2026-10-28, do not change
rng = numpy.random.default_rng(SEED)
rng1, rng2, rng3, rng4, rng5, rng6, rng7, rng8 = rng.spawn(8)

## Step 1: From CDF to density


### 1.1 The four distributions: CDF, pdf, and support

For every continuous random variable the CDF is $F(x) = P(X \le x)$ and the pdf is its derivative, $f(x) = F'(x)$. Probabilities are areas under the pdf: $P(a < X \le b) = F(b) - F(a) = \int_a^b f(x)\,dx$. The **support** is the set of $x$ where $f(x) > 0$. All four use unit parameters, $\lambda = 1$ and $\sigma = 1$.

**Uniform on $[0,1]$.** Support $[0,1]$.
$$F(x)=\begin{cases}0 & x<0\\ x & 0\le x\le 1\\ 1 & x>1\end{cases}\qquad\qquad f(x)=\begin{cases}1 & 0\le x\le 1\\ 0 & \text{otherwise}\end{cases}$$

**Exponential, rate $\lambda$.** Support $[0,\infty)$.
$$F(x)=1-e^{-\lambda x},\qquad f(x)=\frac{d}{dx}\left(1-e^{-\lambda x}\right)=\lambda e^{-\lambda x},\qquad x\ge 0$$

**Standard Gaussian.** Support $(-\infty,\infty)$.
$$F(x)=\Phi(x)=\int_{-\infty}^{x}\frac{1}{\sqrt{2\pi}}e^{-t^2/2}\,dt=\tfrac12\,\mathrm{erfc}\!\left(-\frac{x}{\sqrt2}\right),\qquad f(x)=\varphi(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}$$
$\Phi$ has no elementary closed form, so it is written with the complementary error function $\mathrm{erfc}(z)=\frac{2}{\sqrt\pi}\int_z^\infty e^{-t^2}dt$. Substituting $t = -s/\sqrt2$ turns that integral into the Gaussian one.

**Rayleigh, scale $\sigma$.** Support $[0,\infty)$.
$$F(x)=1-e^{-x^2/(2\sigma^2)},\qquad f(x)=\frac{d}{dx}\left(1-e^{-x^2/(2\sigma^2)}\right)=\frac{x}{\sigma^2}e^{-x^2/(2\sigma^2)},\qquad x\ge 0$$
The derivative uses the chain rule: the inner function $-x^2/(2\sigma^2)$ has derivative $-x/\sigma^2$.

### Measuring agreement: the empirical CDF and the KS statistic

With $n$ samples, the empirical CDF is $\hat F_n(x)=\frac1n\#\{i : X_i\le x\}$. Sorting the samples $x_{(1)}\le\dots\le x_{(n)}$ gives a staircase that jumps from $\frac{i-1}{n}$ to $\frac{i}{n}$ at $x_{(i)}$. The Kolmogorov-Smirnov statistic is the largest vertical gap
$$D_n=\sup_x\left|\hat F_n(x)-F(x)\right| = \max_i\,\max\left(\frac{i}{n}-F(x_{(i)}),\;F(x_{(i)})-\frac{i-1}{n}\right),$$
so `ks_statistic` checks both the top and the bottom of every step. If the samples really come from $F$, then $D_n$ exceeds $1.36/\sqrt n$ only 5 percent of the time. At $n=10^5$ that critical value is $0.0043$.

In [3]:
#Step 1 settings
import numpy
N1 = 10**5                              #Samples per distribution
KS_CRIT_1 = 1.36 / numpy.sqrt(N1)       #95 percent KS critical value at n = 10^5

#Library generator samples for each distribution, drawn from this step's generator
#Each entry holds the samples, the analytic CDF and pdf, and a plotting range
step1 = {
    "Uniform[0,1]": {"samples": rng1.random(N1),
                     "cdf": func.cdf_uniform, "pdf": func.pdf_uniform,
                     "x": numpy.linspace(-0.1, 1.1, 600)},
    "Exponential(1)": {"samples": rng1.exponential(1.0, N1),
                       "cdf": func.cdf_exponential, "pdf": func.pdf_exponential,
                       "x": numpy.linspace(0, 8, 600)},
    "Gaussian(0,1)": {"samples": rng1.standard_normal(N1),
                      "cdf": func.cdf_gaussian, "pdf": func.pdf_gaussian,
                      "x": numpy.linspace(-4.5, 4.5, 600)},
    "Rayleigh(1)": {"samples": rng1.rayleigh(1.0, N1),
                    "cdf": func.cdf_rayleigh, "pdf": func.pdf_rayleigh,
                    "x": numpy.linspace(0, 5, 600)},
}

#Figure 1: CDFs on the top row, densities on the bottom row
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, (name, d) in enumerate(step1.items()):
    xs, ecdf = func.empirical_cdf(d["samples"])

    #Top row: empirical CDF (staircase) against the analytic CDF
    ax = axes[0, col]
    ax.step(xs, ecdf, where="post", lw=2.5, color="tab:blue", label="Empirical CDF")
    ax.plot(d["x"], d["cdf"](d["x"]), "--", lw=1.5, color="black", label="Analytic CDF")
    ax.set_xlim(d["x"][0], d["x"][-1])
    ax.set_title(name + ": CDF")
    ax.set_xlabel("x")
    ax.set_ylabel("F(x)")
    ax.legend(loc="lower right")

    #Bottom row: histogram normalized to a density against the analytic pdf
    ax = axes[1, col]
    ax.hist(d["samples"], bins=100, range=(d["x"][0], d["x"][-1]), density=True,
            color="tab:blue", alpha=0.5, label="Histogram (density)")
    ax.plot(d["x"], d["pdf"](d["x"]), lw=2, color="black", label="Analytic pdf")
    ax.set_title(name + ": pdf")
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.set_ylim(0, 1.3 * ax.get_ylim()[1])     #Headroom so the legend does not cover the bars
    ax.legend(loc="upper right")

fig.suptitle("Figure 1: library samples (n = 10^5) against analytic CDFs and pdfs")
plt.tight_layout()
plt.show()

NameError: name 'rng' is not defined

In [2]:
#Total notebook runtime
total_runtime = time.time() - T_START

print("Total notebook runtime:", round(total_runtime / 60, 2), "minutes")

#Youtube unlisted demo video
# 

Total notebook runtime: 2.29 minutes
